# Customer Churn Prediction — Feature Engineering

**Goal:** Turn raw data into model-ready features without data leakage.

**EDA findings we act on:**
- TotalCharges has 11 hidden blanks → fill with 0
- TotalCharges and tenure are highly correlated (r=0.83) → create `charge_per_service`
- New customers churn most → create `tenure_group`
- Number of services is a strong signal → create `total_services`
- Dataset is imbalanced (26.5% churn) → track this throughout

---
**Notebook structure:**
1. Load & clean raw data
2. Create derived features
3. Validate — no data leakage
4. Build sklearn Pipeline
5. Inspect transformed features
6. Save train/test splits

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('..')   # allow importing from src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

DATA_PATH      = Path('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
PROCESSED_DIR  = Path('../data/processed')
FIGURES_DIR    = Path('../reports/figures')

print('Libraries loaded.')

## 1. Load & Clean Raw Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Raw shape: {df.shape}')

# Fix TotalCharges — stored as string, blanks for tenure=0 customers
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

# Encode target to 0/1
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# Drop customer ID — it carries no predictive signal
df = df.drop(columns=['customerID'])

print(f'Clean shape: {df.shape}')
print(f'Churn rate : {df["Churn"].mean()*100:.1f}%')
df.head(3)

## 2. Create Derived Features

We create these BEFORE the train/test split so the function logic is clear.  
However, **no statistics from the data** (means, medians, frequencies) are used here — only rule-based transformations. That makes them leak-free.

In [ ]:
SERVICE_COLS = [
    'PhoneService', 'MultipleLines', 'InternetService',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]

def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Tenure group — rule-based, no statistics used → leak-free
    df['tenure_group'] = pd.cut(
        df['tenure'],
        bins=[0, 12, 36, np.inf],
        labels=['new', 'mid', 'long']
    ).astype(str)

    # Total active services per customer
    available = [c for c in SERVICE_COLS if c in df.columns]
    df['total_services'] = (df[available] == 'Yes').sum(axis=1)

    # Monthly cost per service — avoids raw multicollinearity between
    # MonthlyCharges and TotalCharges while keeping price-sensitivity signal
    df['charge_per_service'] = df['MonthlyCharges'] / (df['total_services'] + 1)

    # Has any security service (OnlineSecurity OR TechSupport)
    df['has_security'] = (
        (df['OnlineSecurity'] == 'Yes') | (df['TechSupport'] == 'Yes')
    ).astype(int)

    # Is on a long-term contract (one year or two year)
    df['is_long_contract'] = (df['Contract'] != 'Month-to-month').astype(int)

    return df

df = add_derived_features(df)
print('Derived features added:')
print([c for c in df.columns if c not in pd.read_csv(DATA_PATH).columns])

In [ ]:
# Quick sanity check on derived features
df[['tenure', 'tenure_group', 'total_services',
    'MonthlyCharges', 'charge_per_service',
    'has_security', 'is_long_contract']].head(8)

In [ ]:
# Visualise derived features vs churn
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

CHURN_PALETTE = {0: '#4C72B0', 1: '#DD8452'}

# total_services
rates = df.groupby('total_services')['Churn'].mean() * 100
axes[0].bar(rates.index, rates.values, color='steelblue')
axes[0].set_title('Churn Rate by total_services')
axes[0].set_xlabel('total_services')
axes[0].set_ylabel('Churn Rate (%)')
for x, v in zip(rates.index, rates.values):
    axes[0].text(x, v + 0.5, f'{v:.0f}%', ha='center', fontsize=9)

# charge_per_service
df['cps_bin'] = pd.cut(df['charge_per_service'], bins=5)
cps_rates = df.groupby('cps_bin', observed=True)['Churn'].mean() * 100
axes[1].barh([str(b) for b in cps_rates.index], cps_rates.values, color='salmon')
axes[1].set_title('Churn Rate by charge_per_service')
axes[1].set_xlabel('Churn Rate (%)')

# has_security
sec_rates = df.groupby('has_security')['Churn'].mean() * 100
axes[2].bar(['No security', 'Has security'], sec_rates.values,
            color=['#DD8452', '#4C72B0'])
axes[2].set_title('Churn Rate by has_security')
axes[2].set_ylabel('Churn Rate (%)')
for i, v in enumerate(sec_rates.values):
    axes[2].text(i, v + 0.5, f'{v:.1f}%', ha='center')

plt.suptitle('Derived Features vs Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '10_derived_features_churn.png', dpi=150)
plt.show()
df = df.drop(columns=['cps_bin'])

## 3. Train / Test Split

Split **before** fitting any transformer.  
Fitting on the full dataset would leak test-set statistics into the model.

In [ ]:
X = df.drop(columns=['Churn'])
y = df['Churn']

# Stratify keeps churn ratio consistent in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Train: {X_train.shape[0]:,} rows  |  churn rate: {y_train.mean()*100:.1f}%')
print(f'Test : {X_test.shape[0]:,} rows  |  churn rate: {y_test.mean()*100:.1f}%')

## 4. Build the sklearn Pipeline

The pipeline bundles preprocessing and model into one object.  
Calling `.fit()` on training data and `.transform()` on test data keeps everything leak-free.

In [ ]:
NUM_COLS = [
    'tenure', 'MonthlyCharges', 'TotalCharges',
    'total_services', 'charge_per_service',
    'has_security', 'is_long_contract'
]

CAT_COLS = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents',
    'PhoneService', 'MultipleLines', 'InternetService',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod',
    'tenure_group'
]

print(f'Numeric features   : {len(NUM_COLS)} → {NUM_COLS}')
print(f'Categorical features: {len(CAT_COLS)}')

In [ ]:
# Numeric pipeline: fill missing with median, then scale to mean=0 std=1
numeric_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler()),
])

# Categorical pipeline: fill missing, then one-hot encode
# handle_unknown='ignore' means unseen categories in test become all zeros
categorical_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline,    NUM_COLS),
    ('cat', categorical_pipeline, CAT_COLS),
], remainder='drop')

print('Preprocessor built.')

In [ ]:
# Fit ONLY on training data, then transform both splits
preprocessor.fit(X_train)

X_train_transformed = preprocessor.transform(X_train)
X_test_transformed  = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

print(f'Transformed shape — train: {X_train_transformed.shape}')
print(f'Transformed shape — test : {X_test_transformed.shape}')
print(f'Total features after encoding: {len(feature_names)}')

## 5. Validate — No Data Leakage

In [ ]:
# Leakage check 1: scaler statistics must come from train only
# The scaler fitted on train should have slightly different mean/std than full data
scaler = preprocessor.named_transformers_['num'].named_steps['scale']

full_means = X[NUM_COLS].mean()
train_means_raw = X_train[NUM_COLS].mean()
scaler_means = pd.Series(scaler.mean_, index=NUM_COLS)

comparison = pd.DataFrame({
    'full_data_mean' : full_means.round(2),
    'train_mean'     : train_means_raw.round(2),
    'scaler_mean'    : scaler_means.round(2),
    'scaler_==_train': (scaler_means.round(4) == train_means_raw.round(4))
})
print('Scaler was fitted on train data only (scaler_==_train must be True):')
comparison

In [ ]:
# Leakage check 2: numeric features in test should NOT have mean=0 after transform
# (They would be exactly 0 only if test == train, which would indicate leakage)
test_means = pd.Series(
    X_test_transformed[:, :len(NUM_COLS)].mean(axis=0),
    index=NUM_COLS
)
print('Test set means after scaling (should be close to 0 but not exactly 0):')
print(test_means.round(4))

assert not (test_means == 0).all(), 'Leakage detected: test means are exactly 0!'
print('\nLeakage check passed.')

## 6. Inspect Transformed Features

In [ ]:
# Show all feature names after one-hot encoding
feat_df = pd.DataFrame({'feature': feature_names})
feat_df['type'] = feat_df['feature'].apply(
    lambda f: 'numeric' if f.startswith('num__') else 'categorical'
)
print(feat_df.groupby('type')['feature'].count())
print()
print('Categorical features after OHE:')
print(feat_df[feat_df['type'] == 'categorical']['feature'].tolist())

In [ ]:
# Distribution of scaled numeric features on train set
X_train_num = X_train_transformed[:, :len(NUM_COLS)]
num_df = pd.DataFrame(X_train_num, columns=[f.replace('num__', '') for f in feature_names[:len(NUM_COLS)]])

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(num_df.columns):
    axes[i].hist(num_df[col], bins=40, color='steelblue', edgecolor='white')
    axes[i].axvline(0, color='red', linestyle='--', linewidth=1, label='mean=0')
    axes[i].set_title(col)
    axes[i].set_xlabel('Scaled value')

axes[-1].set_visible(False)
plt.suptitle('Scaled Numeric Features — Train Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '11_scaled_numeric_features.png', dpi=150)
plt.show()

In [ ]:
# Correlation of raw numeric features vs derived features with churn
corr_cols = NUM_COLS
corr_with_churn = (
    pd.concat([X_train[NUM_COLS], y_train.reset_index(drop=True)], axis=1)
    .corr()['Churn']
    .drop('Churn')
    .sort_values()
)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#DD8452' if v < 0 else '#4C72B0' for v in corr_with_churn.values]
ax.barh(corr_with_churn.index, corr_with_churn.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlation with Churn')
ax.set_title('Numeric Feature Correlation with Churn (Train set)', fontsize=12)
for i, (name, val) in enumerate(zip(corr_with_churn.index, corr_with_churn.values)):
    ax.text(val + 0.005 if val >= 0 else val - 0.005,
            i, f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '12_feature_correlation_churn.png', dpi=150)
plt.show()

# tenure and is_long_contract show the strongest negative correlation with churn
# charge_per_service and MonthlyCharges show positive correlation

## 7. Save Processed Data

In [ ]:
# Save train/test splits as CSV — preserves column names for debugging
X_train.to_csv(PROCESSED_DIR / 'X_train.csv', index=False)
X_test.to_csv(PROCESSED_DIR  / 'X_test.csv',  index=False)
y_train.to_csv(PROCESSED_DIR / 'y_train.csv', index=False)
y_test.to_csv(PROCESSED_DIR  / 'y_test.csv',  index=False)

# Save fitted preprocessor — this is what we load in the model training notebook
with open(PROCESSED_DIR / 'preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)

print('Saved:')
for p in sorted(PROCESSED_DIR.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size / 1024:.1f} KB)')

## 8. Summary

In [ ]:
print("""
=== FEATURE ENGINEERING SUMMARY ===

DERIVED FEATURES ADDED:
  tenure_group      — new / mid / long  (rule-based, leak-free)
  total_services    — count of 'Yes' service columns
  charge_per_service— MonthlyCharges / (total_services + 1)
  has_security      — 1 if OnlineSecurity OR TechSupport == 'Yes'
  is_long_contract  — 1 if Contract != 'Month-to-month'

PIPELINE STEPS:
  Numeric  → median impute → StandardScaler
  Categorical → mode impute → OneHotEncoder (handle_unknown='ignore')

SPLIT:
  Train: 5,634 rows  (80%)  — churn rate 26.5%
  Test :  1,409 rows (20%)  — churn rate 26.5%
  Stratified split → churn ratio preserved in both sets

LEAKAGE CHECKS:
  Scaler fitted on train only  ✓
  Test means after scaling ≠ 0 ✓

FINAL FEATURE COUNT:
  7 numeric + ~44 OHE categorical = ~51 features total

NEXT STEP:
  03_model_training.ipynb — XGBoost + MLflow + cross-validation
""")